![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 06: Multi-Agent Systems and Safety)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 6A: Multi-Agent Collaboration

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Optional real multi-agent model calls.</td></tr>
<tr><td align="left">Main output</td><td>Design and inspect a local collaboration record with explicit planner, researcher and critic roles.</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m06a-overview)
2. [Setup and Background](#m06a-background)
3. [Core Concepts](#m06a-data)
4. [Guided Implementation](#m06a-workflow)
5. [Testing and Analysis](#m06a-testing)
6. [Student Tasks](#m06a-tasks)
7. [Submission and Reflection](#m06a-submission)

---

<a id="m06a-overview"></a>

### 1. Overview and Learning Goals

This session is **M06A: Multi-Agent Collaboration**. Its role in the unit is to extend the earlier single-agent workflow ideas into a setting where several specialised agents cooperate on one task.

The central theme is:

```text
Design and inspect a local collaboration record with explicit planner, researcher and critic roles.
```

A useful way to picture a multi-agent system is a newsroom. A newspaper does not ask one person to choose the story, gather the facts, check them, and typeset the front page. It uses a planner (the editor who assigns the work), a researcher (the journalist who collects evidence), a critic (the fact-checker who challenges weak claims), and a synthesiser (the sub-editor who turns approved material into the final article). Each person has a clear role boundary, and work moves between them through explicit handoffs. In this session you build the same structure in code, so that every handoff is visible and testable.

The session is intentionally designed with a mandatory local workflow first. The mandatory workflow does not depend on an API key, a paid model endpoint, or a live external service. This matters because the main learning objective is the architecture: how information is represented, passed between roles, checked, and converted into a safe output. Once you can see the architecture clearly in a local simulation, you will recognise it later inside real frameworks such as LangGraph or CrewAI.

The concepts used in this session are:

```text
1. planner
2. researcher
3. critic
4. synthesiser
5. handoff
6. role boundary
```

The general workflow is:

```text
User request
     |
     v
[1] Validate the input ----(unsafe request)----> refuse with a clear reason
     |
     v
[2] Select approved local context
     |
     v
[3] Apply the local workflow logic
     |
     v
[4] Produce a structured result
     |
     v
[5] Inspect the result and its limitations
```

By the end of this session, you should be able to describe the workflow in your own words, run the mandatory local implementation, inspect intermediate outputs, add a small extension, test normal, edge and failure cases, and explain how the design would change if a real model or an agent framework were added.

<a id="m06a-background"></a>

### 2. Setup and Background

#### 2.1 Conceptual Background

The important point in this practical is not just to make a notebook run. The important point is to understand the design discipline behind an agentic AI workflow.

A weak workflow often does this:

```text
User request -----> [ one large prompt ] -----> model output
```

This is simple, but it hides too many decisions. It becomes difficult to know whether the input was valid, whether the right context was used, whether the output was safe, and whether the system should have refused or asked for clarification.

A stronger workflow separates the steps so that each decision is visible and testable:

```text
User request
     |
     v
Input validation ----------- refuses or flags unsafe requests
     |
     v
Context or state selection - only approved data enters the workflow
     |
     v
Controlled transformation -- each step is small and observable
     |
     v
Structured output ---------- carries its own evidence and limitations
     |
     v
Tests and review ----------- normal, edge and failure cases
```

For **Multi-Agent Collaboration**, these concepts matter:

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>How it is used</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">planner</td><td>Decides what happens next and in what order. In this notebook, <code>run_local_workflow</code> is a minimal planner: it fixes the sequence validate, select, build, and stops early when a stage fails.</td></tr>
<tr><td align="left">researcher</td><td>Gathers evidence instead of inventing it. <code>select_relevant_items</code> plays the researcher by scoring approved items against the request.</td></tr>
<tr><td align="left">critic</td><td>Challenges weak results. The insufficient-context branch of <code>build_structured_result</code> is a simple critic: when the evidence is missing, it says so rather than guessing.</td></tr>
<tr><td align="left">synthesiser</td><td>Assembles the final product from approved material only. The completed branch builds the summary strictly from the selected items.</td></tr>
<tr><td align="left">handoff</td><td>The structured dictionary each stage passes to the next. A handoff is a contract: the receiving stage can rely on its shape without re-checking everything.</td></tr>
<tr><td align="left">role boundary</td><td>Each function does exactly one job. The researcher never rewrites the request, and the critic never fetches new data. Clear boundaries are what make failures traceable.</td></tr>
</tbody>
</table>

</div>

In this session the roles form a small newsroom-style pipeline:

```text
                 User request
                      |
                      v
+---------+  plan   +------------+  findings   +--------+
| Planner | ------> | Researcher | ----------> | Critic |
+---------+         +------------+             +--------+
                                                    |
                                     approved or challenged notes
                                                    |
                                                    v
                                             +-------------+
                                             | Synthesiser |
                                             +-------------+
                                                    |
                                                    v
                                      Structured, inspectable answer
```

Every arrow above is a handoff, and every box sees only what it needs. That is the role boundary in action, and it is what makes a multi-agent failure traceable to a single role.

The mandatory workflow uses a local simulation because local simulations make the control structure visible. Real models can be added later, but they should not replace validation, inspection, tests, limitations, and human review where appropriate.

<a id="m06a-setup"></a>

#### 2.2 Environment and Safety

Run the setup cell below before anything else. The mandatory part of this session uses the Python standard library only, so there is nothing to install and no API key to configure. This is deliberate: you should be able to complete the core learning in Google Colab or local Jupyter, on any machine, without spending money or waiting for network access.

When the cell runs correctly you will see `Setup complete.` printed. If you see an error or no output at all, restart the runtime (in Colab: `Runtime > Restart runtime`) and run the notebook again from the top, because every later cell depends on the names defined here.

The safety boundary for this session is:

```text
1. Use only approved public-style or synthetic teaching data.
2. Do not use private documents, credentials, emails, student records or hidden instructor materials.
3. Do not perform real external side effects.
4. Show limitations when the local workflow does not have enough information.
5. Keep output inspectable and testable.
```

Treat this boundary as part of the design rather than an afterthought. Every function you meet below either enforces one of these rules or makes it easy to check that the rules were followed.

In [ ]:
# Standard-library imports only. The mandatory workflow needs no installs,
# no API keys and no network access, so it behaves identically in Colab and
# in local Jupyter - and it still works with the network unplugged.
import json
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

print("Setup complete.")

<a id="m06a-data"></a>

### 3. Core Concepts

#### 3.1 Approved Local Data

The local data below is the approved evidence store for this practical. It is synthetic teaching data: nothing in it is private, and it is deliberately small so that you can read every item and predict, before running any code, which item a given request should select. That predictability is what makes the rest of the notebook debuggable.

Each item has:

```text
item_id: stable identifier used in outputs, so results can cite their evidence
title: short human-readable title
content: approved teaching content the workflow may summarise
tags: labels the selection step can match against a request
risk_level: low / medium / high, a caution label for downstream handling
```

When you run the cell you should see the item count (3) and the first item printed as JSON. Take a minute to read all three items now; every later result in this notebook traces back to them.

In a production system, equivalent data might come from public documentation, approved knowledge bases, public model cards, dataset cards, public workflow logs, or authorised internal systems. This practical does not use those live sources, but the discipline is the same: the workflow may only build answers from evidence that has been explicitly approved.

In [ ]:
# Approved synthetic teaching items for this session.
# Design note: the store is deliberately tiny (three items) so you can trace
# every selection decision by eye. "risk_level" is a caution label available
# to downstream handling; it does not block selection on its own.
LOCAL_ITEMS = [
    {
        "item_id": "M06A-001",
        "title": "Planner Basics",
        "content": "The planner decomposes the request into bounded steps and states what evidence each downstream role needs.",
        "tags": ["planner", "decomposition", "steps", "evidence", "handoff"],
        "risk_level": "low"
    },
    {
        "item_id": "M06A-002",
        "title": "Researcher Practice",
        "content": "The researcher gathers only approved evidence and returns sources and uncertainty rather than an unsupported conclusion.",
        "tags": ["researcher", "approved", "evidence", "sources", "uncertainty"],
        "risk_level": "low"
    },
    {
        "item_id": "M06A-003",
        "title": "Critic Safety",
        "content": "The critic checks the draft against the request and evidence, records defects, and can send work back instead of approving it automatically.",
        "tags": ["critic", "review", "evidence", "handoff", "safety"],
        "risk_level": "medium"
    },
]

print("Number of local items:", len(LOCAL_ITEMS))
print(json.dumps(LOCAL_ITEMS[0], indent=2))

The local items play the same role as a small approved knowledge base, state table, model-card list, evaluation table, or policy scenario list. The purpose is not to cover every real-world case. The purpose is to make the workflow observable: with only three items, you can always work out why a result did or did not include a piece of evidence, which is exactly the habit you will need when the store contains thousands of items.

<a id="m06a-workflow"></a>

### 4. Guided Implementation

#### 4.1 Mandatory Local Workflow

The workflow has four functions:

```text
1. validate_request        - the gatekeeper: is this request safe and well-formed?
2. select_relevant_items   - the evidence gatherer: which approved items match?
3. build_structured_result - the answer builder: assemble, or admit insufficiency
4. run_local_workflow      - the orchestrator: run the stages in a fixed order
```

In newsroom terms, `run_local_workflow` is the editor (planner) who fixes the order of work, `select_relevant_items` is the journalist (researcher) who gathers evidence, the insufficient-context branch of `build_structured_result` is the fact-checker (critic) who blocks unsupported stories, and its completed branch is the sub-editor (synthesiser) who assembles the final piece. Each handoff between them is an explicit structured dictionary, so you can inspect exactly what crossed each role boundary.

One design convention is worth noticing before you read the code: every function returns the same envelope `{"ok": ..., "error": ..., "result": ...}`. Here `ok: False` means the function itself could not do its job (for example, the input was invalid), while a refusal or an insufficient-context outcome is reported inside `result` with `ok: True`, because deciding to refuse is a successful safety decision, not a malfunction. This distinction shows up again in the tests.

The implementation is intentionally explicit rather than compact. In teaching notebooks, readable logic is more valuable than clever one-line code.

In [ ]:
# Text utilities and the input gatekeeper.
#
# Design decision: every function returns the same envelope
# {"ok": ..., "error": ..., "result": ...}. Failures travel as data instead
# of exceptions, so the orchestrator can react to them without try/except.

def normalise_text(text: str) -> str:
    # Lowercase and collapse whitespace so that matching is not fooled by
    # capitalisation or spacing differences.
    return re.sub(r"\s+", " ", text.lower()).strip()


def tokenise(text: str) -> List[str]:
    # Reduce text to plain alphabetic words. Non-string input returns an
    # empty list so downstream scoring degrades safely instead of crashing.
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", normalise_text(text))


def validate_request(request: str) -> Dict[str, Any]:
    # Cheapest check first: an empty or non-string request is a caller
    # error, so the envelope reports ok=False.
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()
    # A deliberately simple, transparent screen for teaching purposes.
    # Production systems layer stronger defences on top (classifiers,
    # allow-lists, instruction hierarchies); the design point here is that
    # screening happens BEFORE the request can influence anything else.
    unsafe_terms = [
        "private file", "password", "api key", "credential", "send email",
        "delete all", "shell command", "student record", "hidden solution"
    ]

    if any(term in lower for term in unsafe_terms):
        # Note ok=True here: refusing an unsafe request is a successful
        # safety decision by the gatekeeper, not a malfunction.
        return {
            "ok": True,
            "error": None,
            "result": {
                "allowed": False,
                "reason": "The request asks for private data, credentials, hidden material or external side effects."
            }
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "allowed": True,
            "reason": "The request is allowed for the local teaching workflow."
        }
    }

In [ ]:
def select_relevant_items(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # Guard the parameter before doing any work: a zero or negative top_k
    # is a caller error, reported through the standard envelope.
    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    # Scoring is plain word overlap between the request and each item's
    # title, content and tags. Design decision: overlap is far weaker than
    # embeddings, but it is fully transparent - you can always explain a
    # score by pointing at the shared words, which is ideal for debugging.
    request_terms = set(tokenise(request))
    scored = []

    for item in items:
        item_text = " ".join([
            item.get("title", ""),
            item.get("content", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(tokenise(item_text))
        score = len(request_terms.intersection(item_terms))
        # Items with no overlap at all are dropped: an unrelated item must
        # never be presented as evidence just to fill the quota.
        if score > 0:
            selected = dict(item)  # copy, so scoring never mutates the store
            selected["score"] = score
            scored.append(selected)

    scored.sort(key=lambda item: item["score"], reverse=True)

    # top_k defaults to 2: small enough that you can read every piece of
    # selected evidence, large enough to observe the ranking behaviour.
    return {"ok": True, "error": None, "result": scored[:top_k]}

In [ ]:
def build_structured_result(request: str, selected_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    # No evidence means no answer. Refusing to invent content when the
    # approved data cannot support it is the single most important habit
    # this workflow teaches.
    if not selected_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "insufficient_context",
                "summary": "The approved local data does not contain enough information to complete this request.",
                "selected_items": [],
                "limitations": [
                    "No sufficiently relevant local item was selected.",
                    "The workflow should not invent missing information."
                ]
            }
        }

    summary = (
        "The workflow selected approved local items and produced a structured result for "
        "Multi-Agent Collaboration. The result is based only on selected local evidence."
    )

    # Even a successful result carries a limitations list. Honest outputs
    # state their own boundaries, so a human reviewer knows what to check.
    return {
        "ok": True,
        "error": None,
        "result": {
            "status": "completed",
            "summary": summary,
            "selected_items": selected_items,
            "limitations": [
                "This is a local teaching workflow, not a live external system.",
                "The result should be checked before being reused in a real setting."
            ]
        }
    }

In [ ]:
def run_local_workflow(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # The orchestrator fixes the order of stages: validate, then select,
    # then build. Each stage's envelope is checked before the next stage
    # runs, so a failure or a refusal stops the pipeline immediately.
    validation = validate_request(request)
    if not validation["ok"]:
        return validation

    if not validation["result"]["allowed"]:
        # A refusal is converted into a normal structured result, so callers
        # handle it like any other outcome - visible, loggable and testable.
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "refused",
                "summary": validation["result"]["reason"],
                "selected_items": [],
                "limitations": ["The request is outside the allowed safety boundary."]
            }
        }

    selected = select_relevant_items(request, items, top_k=top_k)
    if not selected["ok"]:
        return selected

    structured = build_structured_result(request, selected["result"])
    if not structured["ok"]:
        return structured

    # The final envelope echoes the original request, so every result is a
    # self-contained record: request in, evidence used, status, limitations.
    return {
        "ok": True,
        "error": None,
        "result": {
            "request": request,
            **structured["result"]
        }
    }


# One end-to-end run. Expect status "completed" with one or two selected
# items, because this request shares words with the approved items.
example_result = run_local_workflow("How should planner, researcher and critic roles hand off work safely?", LOCAL_ITEMS)
example_result

<a id="m06a-inspection"></a>

#### 4.2 Inspection and Interpretation

The output should be inspected rather than accepted blindly. This is the common evaluation habit across M05–M08: an agentic result is only trustworthy if you can trace it back to its evidence.

You should check:

```text
1. Was the request allowed?
2. Which local items were selected?
3. Are the selected items genuinely relevant, or did they match on incidental words?
4. Did the workflow state limitations?
5. Did it refuse unsafe requests?
```

The helper below prints the result as a short audit view. For the example request you should see status `completed`, one or two selected items with their overlap scores, and a limitations list. If you see `insufficient_context` instead, the request wording no longer overlaps the item text - which is itself a useful lesson in how fragile keyword matching can be.

In [ ]:
def display_workflow_result(result: Dict[str, Any]) -> None:
    # A human-readable audit view: status first, then the evidence that was
    # actually used, then the limitations. Reviewers read this top-down.
    if not result.get("ok"):
        print("ERROR:", result.get("error"))
        return

    payload = result["result"]
    print("Status:", payload.get("status"))
    print("Summary:", payload.get("summary"))

    print("\nSelected items:")
    if not payload.get("selected_items"):
        print("- None")
    for item in payload.get("selected_items", []):
        # item_id and score make each piece of evidence traceable back to
        # the approved store and to the selection decision.
        print(f"- {item['item_id']} | score={item.get('score')} | {item['title']}")
        print(f"  {item['content']}")

    print("\nLimitations:")
    for limitation in payload.get("limitations", []):
        print("-", limitation)


display_workflow_result(example_result)

A strong result is not necessarily the longest result. A strong result is inspectable, grounded in selected items, and clear about limitations. If a selected item is irrelevant - for example, if it matched only on a common word - the final result should not be trusted even though the workflow reports success. Automated checks catch structural problems; judging relevance is still your job.

<a id="m06a-optional"></a>

#### 4.3 Optional Real Model or Package Section

This section is optional. The mandatory workflow already demonstrates the design pattern. A real package or model can be added later, but it must slot into the existing structure rather than replace it: validation, approved context, structured output and inspection all stay.

The correct pattern is:

```text
Validated request
      |
      v
Selected approved context
      |
      v
Model or package call        <-- the only step that changes
      |
      v
Structured result
      |
      v
Inspection and limitations
```

If you extend this session, a natural step is to give each role a real model call: one prompt that plans the steps, one that drafts from the selected items only, and one that critiques the draft against the evidence. Frameworks such as LangGraph and CrewAI industrialise exactly this pattern. Keep each role's input restricted to validated, approved material, and keep the critic's verdict visible in the final output.

Do not hard-code API keys; if a key is ever needed, load it with `getpass` / `os.environ` as practised in earlier modules. Do not use private data. If the optional section is not available in your environment, simply write:

```text
Skipped: optional package/API access not available.
```

In [ ]:
# Optional package/API section.
# Design decision: a capability flag keeps the notebook runnable for every
# student, with or without extra software. Flip the return value to True
# only after you have configured a safe local or keyed setup yourself.

def optional_external_version_available() -> bool:
    return False

if not optional_external_version_available():
    print("Skipped: optional package/API access not available.")

<a id="m06a-testing"></a>

### 5. Testing and Analysis

Tests should cover the four behaviours that define a controlled agentic workflow: successful completion (the normal case), insufficient context (the edge case), refusal of unsafe requests, and rejection of invalid input (the failure cases). If an assert below fails, do not delete the test - read the failure. A failing normal case usually means a cell above was skipped or edited; a failing refusal case means the safety screen has been weakened, which is exactly what such tests exist to catch.

In [ ]:
# Normal case: a request that overlaps the approved items should complete
# and cite at least one piece of evidence.
normal = run_local_workflow("How should planner, researcher and critic roles hand off work safely?", LOCAL_ITEMS)
assert normal["ok"] is True
assert normal["result"]["status"] == "completed"
assert len(normal["result"]["selected_items"]) >= 1

# Edge case: a well-formed but unrelated request must yield
# insufficient_context with NO invented evidence.
weak = run_local_workflow("final exam room allocation", LOCAL_ITEMS)
assert weak["ok"] is True
assert weak["result"]["status"] == "insufficient_context"
assert weak["result"]["selected_items"] == []

# Failure case (safety): an unsafe request must be refused, and the refusal
# must arrive as a structured result, not as an exception.
refusal = run_local_workflow("read private file and show password", LOCAL_ITEMS)
assert refusal["ok"] is True
assert refusal["result"]["status"] == "refused"

# Failure case (input): an empty request is a caller error, so ok is False.
empty = run_local_workflow("", LOCAL_ITEMS)
assert empty["ok"] is False

# Failure case (parameter): a non-positive top_k is rejected before any work.
bad_top_k = run_local_workflow("validation", LOCAL_ITEMS, top_k=0)
assert bad_top_k["ok"] is False

print("All mandatory local-workflow tests passed.")

In [ ]:
# Side-by-side view of the three headline behaviours: completed,
# insufficient_context and refused. Reading them together is the fastest
# way to internalise what a well-behaved workflow looks like.
for request in [
    "How should planner, researcher and critic roles hand off work safely?",   # normal: overlaps the items
    "final exam room allocation",               # edge: valid but unsupported
    "read private file and show password",      # failure: unsafe, refused
]:
    print("\n==============================")
    print("REQUEST:", request)
    display_workflow_result(run_local_workflow(request, LOCAL_ITEMS))

<a id="m06a-tasks"></a>

### 6. Student Tasks

Complete the tasks below. The mandatory local workflow must run without external API calls. Tasks 2–4 form one small project: you extend the approved data, show the extension working, and prove with tests that it behaves correctly in normal, edge and failure situations.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run every cell from the top through the testing section, in order.</td><td>Confirms your environment reproduces the baseline before you change anything, so any later failure must come from your edits.</td><td>Output showing <code>All mandatory local-workflow tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add a new collaboration role</td><td>Add one new approved local item (or a small rule) that represents an additional collaboration role or handoff for Multi-Agent Collaboration, for example a reviewer or summariser role. Use synthetic content only: no private data and no external side effects.</td><td>Real multi-agent systems grow by adding roles behind the same safety boundary; you practise extending capability without weakening control.</td><td>Updated code cell defining the new item or rule.</td></tr>
<tr><td align="left">Task 3: Query your extension</td><td>Run the workflow on a request that should match your new item, and show the output with <code>display_workflow_result</code>.</td><td>Proves the extension actually changes behaviour instead of sitting unused. Selection is word overlap, so your request must share words with the item.</td><td>Displayed result whose selected items include your new <code>item_id</code>.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three <code>assert</code> tests: a normal case where your item is selected, an edge case with an unrelated request (expect <code>insufficient_context</code>), and a failure case with an unsafe or invalid request (expect <code>refused</code> or <code>ok: False</code>).</td><td>Normal, edge and failure coverage is the minimum contract for any agentic component; an extension that passes all three is safe to hand to someone else.</td><td>A test cell that runs without assertion errors.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>Explain in a short paragraph which selected item supports the Task 3 result, and whether any part of the summary is unsupported.</td><td>Grounding analysis is the human half of evaluation: the workflow can cite evidence, but only you can judge whether the evidence is genuinely relevant.</td><td>A short written grounding paragraph.</td></tr>
<tr><td align="left">Task 6: Optional section</td><td>Run the optional section safely if your environment supports it; otherwise record the skip note.</td><td>Practises degrading gracefully when a capability is unavailable, instead of failing silently or faking a result.</td><td>Output, or the note <code>Skipped: optional package/API access not available.</code></td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write 150–250 words on what this workflow teaches about multi-agent collaboration and agentic AI design.</td><td>Explaining a design in your own words is the quickest test of whether you understood it or only executed it.</td><td>A 150–250 word reflection in a markdown cell.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter for M06A (Tasks 2 and 3).
#
# Steps:
# 1. Design one new approved item. Keep the content synthetic and safe.
# 2. Give it a fresh item_id ("M06A-004") so evidence stays traceable.
# 3. Choose tags and content that share words with the request you plan to
#    test: selection is word overlap, so shared words are what get it picked.
# 4. Append it to LOCAL_ITEMS, run the workflow, and display the result.
#
# Uncomment and adapt the example below.

# new_item = {
#     "item_id": "M06A-004",
#     "title": "Human Review Extension",
#     "content": "Human review is important before outputs from Multi-Agent Collaboration are used in real settings.",
#     "tags": ["human_review", "safety", "extension"],
#     "risk_level": "low"
# }
#
# LOCAL_ITEMS.append(new_item)
# result = run_local_workflow("Why is human review important?", LOCAL_ITEMS)
# display_workflow_result(result)

<a id="m06a-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your extension code.
3. Workflow output showing your extension was used.
4. At least three added tests using assert statements.
5. Short grounding or support analysis.
6. Optional package/API result or skipped note.
7. 150–250 word reflection.
```

**Quality checks.** Before submitting, restart the runtime and run the whole notebook from top to bottom (in Colab: `Runtime > Restart and run all`). Confirm that the baseline tests still pass, that your added tests pass, that your new item uses only synthetic content, and that no cell contains an API key, personal data or a private URL. A notebook that only works because of a stale in-memory variable will fail this check - which is exactly what the check is for.

**Debugging guide.** Common problems and their usual causes:

- `NameError` for a function or `LOCAL_ITEMS`: cells were run out of order. Restart and run all cells from the top.
- Your Task 3 request returns `insufficient_context`: the request does not share enough words with your new item. Reword the request, or adjust the item's tags and content - remember that selection is plain word overlap.
- A request is unexpectedly `refused`: it contains one of the screening terms (for example "password" or "api key"). Rephrase the request, and note in your analysis that keyword screens can produce false positives.
- Baseline tests fail after your edits: a shared function's behaviour was changed. Restore the original logic and put your changes in new cells instead.

Reflection questions:

1. What are the main stages of the workflow?
2. Why does the workflow validate input before producing an output?
3. What should happen when there is insufficient approved context?
4. Why should unsafe requests be refused with a structured result rather than answered with a warning?
5. How would the newsroom-style role separation change if one of the roles were a real model call?

#### Further Readings

- https://python.langchain.com/docs/concepts/agents/
- https://langchain-ai.github.io/langgraph/
- https://www.anthropic.com/research/building-effective-agents